# 01 — Data Exploration: NIH Chest X-ray14

This notebook explores the dataset before any training:
- Class distribution and imbalance
- Sample image visualisation
- Patient-level split verification
- Image statistics (pixel mean/std)

**Prerequisite:** Run `bash scripts/download_data.sh` first.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from src.data.dataset import ALL_CLASSES, build_label_matrix

RAW_DIR = '../data/raw/nih-chest-xrays'
PROCESSED_DIR = '../data/processed'

print('Libraries loaded.')

## 1. Load metadata

In [ ]:
df = pd.read_csv(os.path.join(RAW_DIR, 'Data_Entry_2017.csv'))
print(f'Total images : {len(df):,}')
print(f'Total patients: {df["Patient ID"].nunique():,}')
df.head(3)

## 2. Class distribution

In [ ]:
label_matrix = build_label_matrix(df)
class_counts = label_matrix.sum(axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(ALL_CLASSES, class_counts, color='steelblue')
ax.set_xlabel('Number of images')
ax.set_title('NIH Chest X-ray14: Class Distribution')
for bar, count in zip(bars, class_counts):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'{int(count):,}', va='center', fontsize=8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\nClass prevalence (%):')
for cls, cnt in zip(ALL_CLASSES, class_counts):
    print(f'  {cls:<25}: {cnt/len(df)*100:.2f}%')

## 3. Sample X-ray images

In [ ]:
IMAGE_DIR = os.path.join(RAW_DIR, 'images')

# Show one example per class (first occurrence)
n_show = 6
fig, axes = plt.subplots(2, n_show, figsize=(3*n_show, 6))

classes_to_show = ['No Finding', 'Pneumonia', 'Cardiomegaly',
                    'Effusion', 'Pneumothorax', 'Atelectasis']

for col, cls_name in enumerate(classes_to_show):
    cls_idx = ALL_CLASSES.index(cls_name)
    mask = label_matrix[:, cls_idx] == 1
    row_idx = mask.argmax()
    img_file = df.iloc[row_idx]['Image Index']
    img_path = os.path.join(IMAGE_DIR, img_file)
    
    try:
        img = Image.open(img_path).convert('L')
        axes[0, col].imshow(img, cmap='gray')
        axes[0, col].set_title(cls_name, fontsize=9)
        axes[0, col].axis('off')
        
        # Pixel histogram
        axes[1, col].hist(np.array(img).flatten(), bins=50, color='steelblue', alpha=0.7)
        axes[1, col].set_xlabel('Pixel value')
        axes[1, col].set_ylabel('Count')
    except FileNotFoundError:
        axes[0, col].text(0.5, 0.5, 'Image not\nfound', ha='center', va='center',
                           transform=axes[0, col].transAxes)
        axes[0, col].set_title(cls_name)

plt.suptitle('Sample Chest X-rays by Pathology', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Verify patient-level splits

In [ ]:
if os.path.exists(os.path.join(PROCESSED_DIR, 'train.csv')):
    train_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'train.csv'))
    val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, 'val.csv'))
    test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, 'test.csv'))

    # Check no patient appears in two splits
    train_patients = set(train_df['Patient ID'])
    val_patients   = set(val_df['Patient ID'])
    test_patients  = set(test_df['Patient ID'])

    tv_overlap = train_patients & val_patients
    tt_overlap = train_patients & test_patients
    vt_overlap = val_patients   & test_patients

    print(f'Train images   : {len(train_df):>6,} ({len(train_patients):,} patients)')
    print(f'Val   images   : {len(val_df):>6,} ({len(val_patients):,} patients)')
    print(f'Test  images   : {len(test_df):>6,} ({len(test_patients):,} patients)')
    print()
    print(f'Train ∩ Val  patient overlap: {len(tv_overlap)}  (should be 0)')
    print(f'Train ∩ Test patient overlap: {len(tt_overlap)}  (should be 0)')
    print(f'Val   ∩ Test patient overlap: {len(vt_overlap)}  (should be 0)')
else:
    print('Splits not yet created. Run: python src/data/preprocess.py')

## 5. Image statistics (mean / std for normalisation)

In [ ]:
import torchvision.transforms as T
import torch

if os.path.exists(os.path.join(PROCESSED_DIR, 'train.csv')):
    # Sample 500 images to estimate pixel statistics
    sample_df = train_df.sample(min(500, len(train_df)), random_state=42)
    to_tensor = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor()])
    
    pixels = []
    for img_name in sample_df['Image Index']:
        try:
            img = Image.open(os.path.join(IMAGE_DIR, img_name)).convert('L')
            t = to_tensor(img)
            pixels.append(t)
        except FileNotFoundError:
            continue
    
    if pixels:
        all_pixels = torch.stack(pixels)
        print(f'Mean : {all_pixels.mean():.4f}')
        print(f'Std  : {all_pixels.std():.4f}')
        print('(These values are used in the normalise_mean/std config fields)')
else:
    print('Run preprocessing first.')